## Imports

In [ ]:
import os, glob, json, re
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.optim as optim

## Load saved layers

In [ ]:
def load_npz(path):
    z = np.load(path, allow_pickle=True)
    X = z["X"]
    concepts = z["concepts"]
    surfaces = z["surfaces"]
    meta = json.loads(str(z["meta"].item())) if z["meta"].dtype == object else json.loads(str(z["meta"]))
    return X, concepts, surfaces, meta

In [ ]:
model_type = "robustness" # C_H (main/siblings/robustness)

all_paths = sorted(glob.glob("invariance_outputs_nb/*/*/*/*.npz"))
all_paths = [p for p in all_paths if model_type in p]

dolly15k_dataset_paths = [p for p in all_paths if "dolly15k" in p]
dolly2k_dataset_paths = [p for p in all_paths if "dolly2k" in p]
custom_dataset_paths = [p for p in all_paths if "custom" in p]

paths = dolly15k_dataset_paths # C_H

rows = []
blocks = []
for p in paths:
    X, c, s, m = load_npz(p)
    model = m["model_id"]
    d = X.shape[1]
    rows.append({
        "path": p,
        "model": model,
        "layer": m["layer_index_0based"]+1,
        "N": X.shape[0],
        "d": d
    })
    blocks.append((X, c, s, model, m))
manifest = pd.DataFrame(rows)
display(manifest)

In [ ]:
if model_type == "main":
    MODEL_IDS = [
        "mistralai/Mistral-7B-Instruct-v0.3",
        "meta-llama/Meta-Llama-3-8B-Instruct",
        "google/gemma-7b-it",
        "Qwen/Qwen2.5-7B-Instruct",
        # "openai-community/gpt2-xl",
        "zai-org/glm-4-9b-chat-hf",
        "deepseek-ai/deepseek-moe-16b-chat",
        "internlm/internlm2-chat-7b",
        "microsoft/Phi-4-mini-instruct",
        "tiiuae/Falcon3-7B-Instruct",
        # "baichuan-inc/Baichuan2-13B-Chat"
    ]
elif model_type == "siblings":
    MODEL_IDS = [
        "mistralai/Mistral-7B-Instruct-v0.1",
        "meta-llama/Llama-3.2-1B-Instruct",
        "google/gemma-2-2b-it",
        "Qwen/Qwen2.5-3B-Instruct",
        # "openai-community/gpt2",
        "zai-org/glm-4-9b-chat",
        "deepseek-ai/deepseek-llm-7b-chat",
        "internlm/internlm2-chat-1_8b",
        "microsoft/Phi-3-mini-4k-instruct",
        "tiiuae/Falcon3-3B-Instruct",
        # "baichuan-inc/Baichuan2-7B-Chat"
    ]

In [ ]:
MODEL_NAME_MAP = {
    "mistralai/Mistral-7B-Instruct-v0.3": "Mistral-7B-Instruct-v0.3",
    "meta-llama/Meta-Llama-3-8B-Instruct": "Meta-Llama-3-8B-Instruct",
    "google/gemma-7b-it": "Gemma-7B-it",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen2.5-7B-Instruct",
    "openai-community/gpt2-xl": "GPT2-xl",
    "zai-org/glm-4-9b-chat-hf": "GLM-4-9B-chat-hf",
    "deepseek-ai/deepseek-moe-16b-chat": "Deepseek-moe-16B-chat",
    "internlm/internlm2-chat-7b": "Internlm2-chat-7B",
    "tiiuae/Falcon3-7B-Instruct": "Falcon3-7B-Instruct",
    "microsoft/Phi-4-mini-instruct": "Phi-4-mini-Instruct",
    "baichuan-inc/Baichuan2-13B-Chat": "Baichuan2-13B-Chat",

    "mistralai/Mistral-7B-Instruct-v0.1": "Mistral-7B-Instruct-v0.1",
    "meta-llama/Llama-3.2-1B-Instruct": "Llama-3.2-1B-Instruct",
    "google/gemma-2-2b-it": "Gemma-2-2B-it",
    "Qwen/Qwen2.5-3B-Instruct": "Qwen2.5-3B-Instruct",
    "openai-community/gpt2": "GPT2",
    "zai-org/glm-4-9b-chat": "GLM-4-9B-chat",
    "deepseek-ai/deepseek-llm-7b-chat": "Deepseek-llm-7B-chat",
    "internlm/internlm2-chat-1_8b": "Internlm2-chat-1_8B",
    "tiiuae/Falcon3-3B-Instruct": "Falcon3-3B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct": "Phi-3-mini-4k-Instruct",
    "baichuan-inc/Baichuan2-7B-Chat": "Baichuan2-7B-Chat",

    "mistralai/Mistral-7B-Instruct-v0.2": "Mistral-7B-Instruct-v0.2",
    "Qwen/Qwen1.5-1.8B-Chat": "Qwen1.5-1.8B-Chat",
}

In [ ]:
# BREAKPOINT_0

## Classification

In [ ]:
# -------------------------------------------------
# GPU-backed classifier on JL features
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------- Hyperparams ----------
K_COMMON     = 256       # JL common dimension
RP_SEED      = 123
N_SPLITS     = 5
RANDOM_STATE = 42
USE_CLASS_WEIGHT = True  # helpful if models are imbalanced
LR           = 1e-2
EPOCHS       = 300       # small dataset, so this is fine

# ---------- Helper: JL projection per block ----------
def jl_project_block(X, k=K_COMMON, seed=RP_SEED):
    rp = GaussianRandomProjection(n_components=k, random_state=seed)
    return rp.fit_transform(X.astype(np.float64))

# ---------- Build dataset from blocks (with layer tracking) ----------
Xs, y_models, y_concepts, y_layers = [], [], [], []

for item in blocks:
    if len(item) == 5:
        X, concepts, surfaces, model_label, meta = item
    elif len(item) == 4:
        X, concepts, surfaces, model_label = item
        meta = {}
    else:
        raise ValueError("Unexpected blocks tuple shape; expected 4 or 5 elements per block.")

    Xp = jl_project_block(X)                     # [N, K_COMMON]
    layer_idx = meta.get("layer_index_0based", -1)

    Xs.append(Xp)
    y_models.append(np.array([model_label] * len(concepts), dtype=object))
    y_concepts.append(concepts.astype(int))      # group key for leakage-safe splits
    y_layers.append(np.full(len(concepts), layer_idx, dtype=int))

X_all      = np.vstack(Xs)                       # [M, K_COMMON]
y_model    = np.concatenate(y_models)            # [M] object labels (strings)
group_ids  = np.concatenate(y_concepts)          # [M] ConceptID (for GroupKFold)
layer_ids  = np.concatenate(y_layers)            # [M] layer index (0-based)

# Standardize globally (after JL)
scaler = StandardScaler(with_mean=True, with_std=True)
X_all = scaler.fit_transform(X_all)

# Encode model labels to integers for the classifier
classes, y_int = np.unique(y_model, return_inverse=True)
num_classes = len(classes)
print("Model classes detected:", classes.tolist())

# ---------- Group-aware CV (held-out prompts) ----------
gkf = GroupKFold(n_splits=N_SPLITS)
fold_rows = []
y_true_all, y_pred_all = [], []
results_rows = []   # per-example results to analyze per-layer

class LinearClassifier(nn.Module):
    """Simple multinomial logistic regression: linear layer + softmax via CrossEntropyLoss."""
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(in_dim, num_classes)
    def forward(self, x):
        return self.linear(x)

for fold, (tr, te) in enumerate(gkf.split(X_all, y_int, groups=group_ids), start=1):
    print(f"\n=== Fold {fold} ===")

    # Split numpy arrays
    Xtr_np, Xte_np = X_all[tr], X_all[te]        # [N_tr, 256], [N_te, 256]
    ytr_np, yte_np = y_int[tr], y_int[te]        # [N_tr], [N_te]

    # metadata for test rows
    true_model_names = y_model[te]               # string labels
    layer_te         = layer_ids[te]             # layer indices

    # Convert to torch tensors on GPU/CPU
    Xtr = torch.tensor(Xtr_np, dtype=torch.float32, device=device)
    Xte = torch.tensor(Xte_np, dtype=torch.float32, device=device)
    ytr = torch.tensor(ytr_np, dtype=torch.long, device=device)
    yte = torch.tensor(yte_np, dtype=torch.long, device=device)

    # Build model
    model = LinearClassifier(K_COMMON, num_classes).to(device)

    # Class weights (to mimic class_weight="balanced")
    if USE_CLASS_WEIGHT:
        class_counts = np.bincount(ytr_np, minlength=num_classes).astype(np.float32)
        class_counts[class_counts == 0] = 1.0
        weights = (len(ytr_np) / (num_classes * class_counts))
        class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=LR)

    # ---- Train loop ----
    model.train()
    for epoch in range(EPOCHS):
        optimizer.zero_grad()
        logits = model(Xtr)          # [N_tr, num_classes]
        loss = criterion(logits, ytr)
        loss.backward()
        optimizer.step()

    # ---- Evaluate on this fold ----
    model.eval()
    with torch.no_grad():
        logits_te = model(Xte)       # [N_te, num_classes]
        yhat_te  = torch.argmax(logits_te, dim=1)

    # Move back to CPU for sklearn metrics
    yte_cpu   = yte.cpu().numpy()
    yhat_cpu  = yhat_te.cpu().numpy()

    acc  = accuracy_score(yte_cpu, yhat_cpu)
    f1m  = f1_score(yte_cpu, yhat_cpu, average="macro")
    f1w  = f1_score(yte_cpu, yhat_cpu, average="weighted")
    fold_rows.append({"fold": fold, "acc": acc, "macro_f1": f1m, "weighted_f1": f1w})

    y_true_all.append(yte_cpu)
    y_pred_all.append(yhat_cpu)

    # per-example results for this fold
    results_rows.append(pd.DataFrame({
        "fold": fold,
        "true_idx": yte_cpu,
        "pred_idx": yhat_cpu,
        "true_model": true_model_names,
        "pred_model": classes[yhat_cpu],
        "layer_index_0based": layer_te,
    }))

fold_df = pd.DataFrame(fold_rows)
print("\nFold metrics:")
display(fold_df)
print("CV mean ± std:")
print(fold_df.agg(["mean","std"])[["acc","macro_f1","weighted_f1"]])

y_true_all = np.concatenate(y_true_all)
y_pred_all = np.concatenate(y_pred_all)

print("\nClassification report (aggregated across folds):")
print(classification_report(y_true_all, y_pred_all, target_names=classes))

In [ ]:
results_df = pd.concat(results_rows, ignore_index=True)
results_df["correct"] = (results_df["true_idx"] == results_df["pred_idx"])

layer_stats = (
    results_df
    .groupby(["true_model", "layer_index_0based"])
    .agg(
        total=("correct", "size"),
        correct=("correct", "sum")
    )
    .reset_index()
)
layer_stats["accuracy"] = layer_stats["correct"] / layer_stats["total"]

print("\nPer-layer stats (head):")
display(layer_stats.head())

In [ ]:
plt.rcParams.update({'font.size': 20})  # Change 16 to your desired base font size
sns.set_context("paper", font_scale=2.0)  # Adjust font_scale as needed

# Map class names to numbers 1–9
classes_pretty = np.array([MODEL_NAME_MAP.get(c, c) for c in classes])
class_numbers = np.arange(1, len(classes_pretty) + 1)
number_to_class = dict(zip(class_numbers, classes_pretty))

# Compute confusion matrix
cm_raw = confusion_matrix(y_true_all, y_pred_all, labels=np.arange(len(classes)))
cm = cm_raw.astype(float)
cm = cm / cm.sum(axis=1, keepdims=True)
cm_df = pd.DataFrame(cm, index=class_numbers, columns=class_numbers)

plt.figure(figsize=(8, 12))
ax = sns.heatmap(
    cm_df,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0.0, vmax=1.0,
    cbar=False,
    xticklabels=class_numbers,
    yticklabels=class_numbers
)

plt.xticks(ha="center")
plt.xlabel("Predicted")
plt.ylabel("True")

# Create legend as a 3x3 grid below the heatmap
legend_labels = [f"{num}: {name}" for num, name in number_to_class.items()]
legend_grid = np.array(legend_labels).reshape(3, 3)

# Add legend as a table below the plot
table = plt.table(
    cellText=legend_grid,
    cellLoc='left',
    loc='bottom',
    bbox=[-0.05, -0.3, 1.1, 0.18]  # [left, bottom, width, height]
)

table.auto_set_font_size(False)
table.set_fontsize(18)  # Smaller font for compactness


# Remove grid lines from the legend table
for key, cell in table.get_celld().items():
    cell.set_edgecolor('none')
    cell.PAD = 0.05

plt.tight_layout(rect=[0, 0.13, 1.5, 1])  # [left, bottom, right, top]
plot_path = f"invariance_outputs_nb/{model_type}/confusion_heatmap_normalized.png"
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
def slugify(text: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", text)


def _first_hit(patterns):
    for p in patterns:
        hits = glob.glob(p)
        if hits:
            return hits[0]
    return None


def _add_layer_if_missing(df):
    if "Layer" not in df.columns:
        df = df.copy()
        df["Layer"] = np.arange(1, len(df) + 1)
    return df


def load_metrics_for_model(model_id, base_dir, use_subdir=True):
    """Load IFAC CSV for a given model."""
    slug = slugify(model_id)
    root = os.path.join(base_dir, slug) if use_subdir else base_dir
    roots = [root, base_dir]

    ifac_path = _first_hit([os.path.join(r, f"{slug}_ifac_k1_k2_k3.csv")
                            for r in roots])

    dfs = {}
    paths = {"IFAC": ifac_path}

    if ifac_path and os.path.exists(ifac_path):
        dfs["IFAC"] = _add_layer_if_missing(pd.read_csv(ifac_path))

    return {"slug": slug, "model_id": model_id, "dfs": dfs, "paths": paths}


def compute_ifac_ylims(entries):
    """Compute global y-limits for IFAC means across all models."""
    vals = []

    for e in entries:
        df = e["dfs"].get("IFAC", None)
        if df is None:
            continue
        cols = [c for c in df.columns
                if c.startswith("IFAC_") and c.endswith("_Mean")]
        if cols:
            vals.append(df[cols].to_numpy().ravel())

    if not vals:
        return (0.0, 1.0)

    arr = np.concatenate(vals)
    vmin, vmax = np.nanmin(arr), np.nanmax(arr)

    if not (np.isfinite(vmin) and np.isfinite(vmax)):
        return (0.0, 1.0)

    pad = 0.05 * (vmax - vmin) if vmax > vmin else 0.05
    return (vmin - pad, vmax + pad)

In [ ]:
entries = [load_metrics_for_model(m, f"invariance_outputs_nb/{model_type}") for m in MODEL_IDS]
ifac_ylim = compute_ifac_ylims(entries)

In [ ]:
def plot_ifac_vs_accuracy_grid(entries, layer_stats, ylims=None, out_pdf=None):
    n_models = len(entries)
    n_rows, n_cols = 3, 3
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4.2 * n_cols, 3 * n_rows),
        constrained_layout=True
    )
    axes = axes.flatten()

    # IFAC keys (same logic as original)
    first_ifac = entries[0]["dfs"].get("IFAC")
    if first_ifac is None:
        raise ValueError("No IFAC data found in first entry.")

    keys = [c for c in first_ifac.columns
            if c.startswith("IFAC_") and c.endswith("_Mean")]
    keys = sorted(keys, key=lambda z: int(z.split("_")[1]))
    legend_labels_ifac = [f"k={k.split('_')[1]}" for k in keys]

    # classification data: pretty model names + 1-based layer index
    ls = layer_stats.copy()
    ls["Models"] = (
        ls["true_model"]
        .map(MODEL_NAME_MAP)
        .fillna(ls["true_model"])
    )
    ls["Layer"] = ls["layer_index_0based"] + 1

    # palette for IFAC curves; accuracy in black
    palette_ifac = sns.color_palette("deep", n_colors=len(keys))
    acc_color = "black"

    for i, entry in enumerate(entries):
        ax = axes[i]
        raw_label = entry["model_id"]
        pretty_label = MODEL_NAME_MAP.get(raw_label, raw_label.split("/")[-1])

        df_ifac = entry["dfs"].get("IFAC", None)
        if df_ifac is None:
            ax.set_visible(False)
            continue

        x = df_ifac["Layer"].to_numpy()

        # IFAC curves (unchanged from original, just pulling colors from palette_ifac)
        for j, key in enumerate(keys):
            ax.plot(
                x,
                df_ifac[key].to_numpy(),
                lw=2,
                label=legend_labels_ifac[j],
                color=palette_ifac[j]
            )

        # classification accuracy overlay
        sub_cls = ls[ls["Models"] == pretty_label]
        if not sub_cls.empty:
            sub_cls = sub_cls.sort_values("Layer")
            x_acc = sub_cls["Layer"].to_numpy()
            y_acc = sub_cls["accuracy"].to_numpy()
            ax.plot(
                x_acc,
                y_acc,
                lw=2.5,
                color=acc_color,
                linestyle="-",
                label="Accuracy"
            )

        if ylims is not None:
            ax.set_ylim(*ylims)

        n_layers = len(x)
        if n_layers >= 3:
            tick_pos = [0, n_layers // 2, n_layers - 1]
            tick_labels = [str(df_ifac["Layer"].iloc[idx]) for idx in tick_pos]
        else:
            tick_pos = np.arange(n_layers)
            tick_labels = [str(l) for l in df_ifac["Layer"]]

        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_labels)
        ax.set_title(pretty_label,
                     fontsize=plt.rcParams["axes.labelsize"],
                     pad=8)
        ax.margins(x=0.01)
        ax.set_xlabel("")
        ax.set_ylabel("")

        # identical y-tick hiding logic as original
        if i % n_cols != 0:
            ax.set_yticklabels([])

        if i == 3:
            ax.set_ylabel("Score / Accuracy",
                          fontsize=plt.rcParams["axes.labelsize"] + 2)
        if i == 7:
            ax.set_xlabel("Layers",
                          fontsize=plt.rcParams["axes.labelsize"] + 2)

    # hide any unused subplots (unchanged)
    for j in range(n_models, n_rows * n_cols):
        axes[j].set_visible(False)

    # Global legend: IFAC ks + Accuracy
    handles, labs = axes[0].get_legend_handles_labels()
    # no dedup needed unless you see duplicates; this matches original style more closely
    fig.legend(
        handles,
        labs,
        loc="lower center",
        ncol=len(labs),
        frameon=False,
        bbox_to_anchor=(0.5, 1.0),
        fontsize=22
    )

    if out_pdf:
        fig.savefig(out_pdf, bbox_inches="tight")
        print("Saved:", out_pdf)

    plt.grid(True)
    plt.show()

In [ ]:
sns.reset_defaults()
plt.rcdefaults()
sns.set_theme(style="whitegrid", context="talk", font_scale=1.1)

plt.rcParams.update({
    "figure.dpi": 120,
    "legend.fontsize": 12,
    # "axes.labelsize": 12
})

plot_ifac_vs_accuracy_grid(
    entries,
    layer_stats,
    ylims=ifac_ylim,
    out_pdf=f"invariance_outputs_nb/{model_type}/ifac_vs_accuracy_grid.pdf"
)

## t-SNE

In [ ]:
BREAKPOINT_1

In [ ]:
k = 256
rng_seed = 42

def project_to_common(X, k, random_state):
    rp = GaussianRandomProjection(n_components=k, random_state=random_state)
    return rp.fit_transform(X.astype(np.float64))

Xk_list, C_list, S_list, M_list = [], [], [], []
for X, c, s, model, meta in blocks:
    if not np.all(np.isfinite(X)):
        print(f"[SKIP] Non-finite values in block for model={model}, layer={meta.get('layer_index_0based')}")
        continue

    Xk = project_to_common(X, k, random_state=rng_seed)
    Xk_list.append(Xk)
    C_list.append(c)
    S_list.append(s)
    M_list.append(np.array([model]*len(c), dtype=object))
    
Xk_all      = np.vstack(Xk_list)
concept_all = np.concatenate(C_list)
surface_all = np.concatenate(S_list)
model_all   = np.concatenate(M_list)

In [ ]:
# global standardize
Xk_all = StandardScaler(with_mean=True, with_std=True).fit_transform(Xk_all)

from openTSNE import TSNE

tsne = TSNE(
    n_components=2,
    perplexity=10,
    random_state=0,
    n_jobs=8
)
Xts = tsne.fit(Xk_all)

df_tsne = pd.DataFrame({
    "Dimension 1": Xts[:,0],
    "Dimension 2": Xts[:,1],
    "ModelRaw": model_all
})

df_tsne["Model"] = df_tsne["ModelRaw"].map(MODEL_NAME_MAP).fillna(df_tsne["ModelRaw"])

In [ ]:
sns.reset_defaults()
plt.rcdefaults()
sns.set_context("paper", font_scale=1.8)

plt.figure(figsize=(12,12))
sns.scatterplot(
    data=df_tsne,
    x="Dimension 1",
    y="Dimension 2",
    hue="Model",
    palette="tab10",
    s=40,
    alpha=0.8
)

plt.title("t-SNE")
plt.grid(False)

# Place legend at bottom, 2 rows, 3 items per row
plt.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, -0.22),   # push below figure
    ncol=3,                        # 3 per row
    frameon=False,
    fontsize=18,
    borderpad=0.0
)

plt.tight_layout()

plot_path = f"invariance_outputs_nb/{model_type}/t-SNE.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()